# 03 · GNN vs Conditional VAE head-to-head

**Problem.** Does message passing over a biological graph actually help, or can a graph-free
conditional VAE (CPA/scGen-style) match it? This is the core ablation behind `graph-perturb`.

**Approach.** On the **same data and the same split**, train two models that share the
`PerturbationModel` interface:
* `GraphSAGEPerturbation` — message passing over a GO BP graph (`requires_graph=True`);
* `ConditionalVAE` — encodes baseline expression + a perturbation latent, no graph
  (`requires_graph=False`).

Both are evaluated with the identical `evaluate_model` path and combined with
`evaluate.compare_models`.

**What to look at.** The two-row-per-split table. The graph prior tends to help most on the harder
`test_combo` split (unseen *double* perturbations), where the VAE has no structural signal to
extrapolate from.

In [ ]:
import logging
import numpy as np
logging.basicConfig(level=logging.WARNING)
np.random.seed(0)

from graph_perturb.config import DataConfig, SplitConfig, ModelConfig, TrainConfig, EvalConfig
from graph_perturb.data import make_splits
from graph_perturb.data.norman import load_norman, make_synthetic_norman
from graph_perturb.data.dataset import build_dataloaders
from graph_perturb.graphs.registry import get_graph_source
from graph_perturb.models import build_model
from graph_perturb.train import train_model
from graph_perturb.evaluate import evaluate_model, compare_models

## 1. Shared data, split, and graph

The VAE ignores the graph, but the GNN needs it; we build one GO graph and reuse the same `splits`
and dataloaders structure for both models so the comparison is apples-to-apples.

In [ ]:
try:
    data = load_norman(DataConfig(name="norman", n_top_genes=2000))
    MODE = "REAL Norman Perturb-seq"
except Exception as exc:
    print(f"[fallback] real Norman load failed ({type(exc).__name__}: {exc})")
    data = make_synthetic_norman(n_genes=80, n_conditions=30, seed=0)
    MODE = "SYNTHETIC offline stand-in"
print(f"DATA MODE: {MODE}  |  cells={data.n_cells} genes={data.n_genes}")

splits = make_splits(data, SplitConfig(test_single_frac=0.3, test_combo_frac=0.3, seed=0))
print({k: len(v) for k, v in splits.as_dict().items()})

graph = get_graph_source("go_bp").build(data.gene_names, use_cache=False)
print(graph)

## 2. Train the GNN and the VAE

`build_model` dispatches on `model_cfg.name`. The dataloaders carry the graph topology; the VAE
simply ignores `edge_index` and reads the dense baseline + perturbation flag.

In [ ]:
train_cfg = TrainConfig(epochs=6, batch_size=16, lr=1e-3, device="cpu",
                        early_stop_patience=6, log_every=100, seed=0)
eval_cfg = EvalConfig(overlap_k=20, splits=("test_single", "test_combo"))
loaders = build_dataloaders(data, graph, splits, train_cfg)

model_cfgs = {
    "gnn": ModelConfig(name="gnn", hidden_dim=32, n_layers=2, dropout=0.1, attention_heads=2),
    "vae": ModelConfig(name="vae", hidden_dim=64, n_layers=2, dropout=0.1, latent_dim=16, kl_weight=1e-3),
}

results_by_model = {}
for name, mcfg in model_cfgs.items():
    print(f"=== training {name} ===")
    # The VAE is graph-free, so build_model ignores the graph for it (we pass None).
    # The *dataset/eval* path, however, always needs the graph to supply edge_index
    # and the one-hot perturbation flag, so evaluate_model uses `graph` for both models.
    g = graph if name == "gnn" else None
    model = build_model(mcfg, num_genes=data.n_genes, graph=g)
    print(f"  requires_graph={model.requires_graph}")
    train_model(model, loaders, train_cfg, ckpt_dir=None)
    results_by_model[name] = evaluate_model(model, data, graph, splits, eval_cfg)


## 3. Head-to-head metrics table

In [ ]:
table = compare_models(results_by_model)
print(table.to_string())
table

## Discussion

* **Where the graph helps.** The GNN propagates a perturbation flag along curated gene-gene edges, so
  activating a transcription factor lights up its known targets even for perturbations unseen in
  training. This structural inductive bias is most valuable on `test_combo`, where two perturbed
  genes' neighborhoods overlap in ways a flat model cannot infer.
* **Where the VAE is competitive.** For `test_single` with abundant similar training conditions, the
  VAE's latent additive composition can capture the dominant response without a graph, and it is
  cheaper to train.
* **Caveat.** These are deliberately short CPU runs; absolute numbers are not publication-grade.
  Increase `epochs`, use the real Norman data, and average over `seed`s before drawing conclusions.
  The infrastructure (shared split, identical eval path) is what makes the comparison trustworthy.